# LegacyAgent V1 - Day 7 - Post-Training Quantization + Re-Eval
(named 08 instead of 07 to keep the notebooks in actual chronological order -- 07 is the biasing baseline)

Goal for today: quantize the fine-tuned model and re-run the exact same frozen eval, to see whether the fine-tuning gains hold up under quantization for edge deployment.

I made two calls here that aren't just copied from earlier notebooks, so worth noting:

1. **Adapter path fix.** Day 6's `ADAPTER_DIR` was pointing at `checkpoints/qlora_v1_run2/final_adapter` -- that's the broken, pre-EOS-fix adapter. The one that actually works (7.36% WER / 22.99% entity WER, confirmed by testing it manually) is at `checkpoints/qlora_v1_run2_eosfix/final_adapter`, so that's what this notebook uses.

2. **Going with int8, merge-then-quantize.** MD just said "int8/int4" without picking one. Day 6's setup -- LoRA adapter on top of a 4-bit base via PEFT -- isn't really the same as a properly quantized, standalone deployable model. So here I load the base model in full bf16, merge the adapter in permanently (`merge_and_unload()`), save that merged model, then reload it in 8-bit (`BitsAndBytesConfig(load_in_8bit=True)`) for the eval. Went with int8 over int4 since it's the more common edge-deployment tier and it's meaningfully different from Day 6's setup. Switching to int4 later would just mean changing the quantization config cell.

In [1]:
# CELL 1
# Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/legacyagent")
EVAL_DIR = ROOT / "eval"

# Corrected adapter path -- see markdown note above.
ADAPTER_DIR = ROOT / "checkpoints" / "qlora_v1_run2_eosfix" / "final_adapter"
MERGED_MODEL_DIR = ROOT / "checkpoints" / "qlora_v1_run2_eosfix" / "merged_model"
RESULTS_DIR = ROOT / "results" / "day7_quantization"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert ADAPTER_DIR.exists(), f"Adapter not found: {ADAPTER_DIR}"
print("Adapter path   :", ADAPTER_DIR)
print("Merged model to:", MERGED_MODEL_DIR)
print("Results dir    :", RESULTS_DIR)

Mounted at /content/drive
Adapter path   : /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/final_adapter
Merged model to: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/merged_model
Results dir    : /content/drive/MyDrive/legacyagent/results/day7_quantization


In [2]:
# CELL 2
# Install pinned dependencies
!pip -q uninstall -y peft transformers torchao
!pip -q install \
    transformers==5.13.0 \
    accelerate \
    peft \
    torchao \
    bitsandbytes \
    librosa \
    soundfile \
    jiwer

import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
assert transformers.__version__ == "5.13.0", transformers.__version__
print("Versions locked correctly")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00


transformers: 5.13.0
peft: 0.20.0
Versions locked correctly


In [3]:
# CELL 3
# Frozen text normalization + WER evaluation (verbatim from Day 2 Cell 4)
import re
import unicodedata
from jiwer import wer

def normalize_for_wer(text: str) -> str:
    """
    Frozen LegacyAgent WER normalization.
    Do not modify after baseline results are produced.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[‐-‒–—−]+", " ", text)
    text = re.sub(r"[^\w\s']", " ", text)
    text = text.replace("'", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Frozen normalizer loaded")

Frozen normalizer loaded


In [4]:
# CELL 4
# Load the frozen Day 2 test benchmark -- same 5 held-out cases
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_benchmark = load_jsonl(EVAL_DIR / "frozen_test_benchmark.jsonl")

print("Test segments:", len(test_benchmark))
print("Cases:", sorted(set(row["case_id"] for row in test_benchmark)))

Test segments: 1163
Cases: ['1996_96-318', '2000_99-1977', '2008_07-1015', '2013_13-115', '2016_15-118']


In [5]:
# CELL 5
# Load base model at full bf16 precision (NOT quantized yet) and attach the
# fine-tuned adapter, so the adapter can be permanently merged.
#
# FIX: skip loading the full bf16 base model entirely if merged_model
# already exists on Drive -- no point loading/attaching just to delete
# it again in Cell 6.
import torch
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

qwen_processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

if not (MERGED_MODEL_DIR / "model.safetensors").exists():
    base_model_fp = Qwen3ASRForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    adapter_model = PeftModel.from_pretrained(base_model_fp, str(ADAPTER_DIR))
    print("Base model + adapter loaded at bf16, ready to merge")
else:
    print("Merged model already exists at:", MERGED_MODEL_DIR)
    print("Skipping bf16 base model load and adapter attach -- not needed.")

processor_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Merged model already exists at: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/merged_model
Skipping bf16 base model load and adapter attach -- not needed.


In [6]:
# CELL 6 - updated
if not (MERGED_MODEL_DIR / "model.safetensors").exists():
    merged_model = adapter_model.merge_and_unload()

    MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(MERGED_MODEL_DIR))
    qwen_processor.save_pretrained(str(MERGED_MODEL_DIR))

    print("Merged model saved to:", MERGED_MODEL_DIR)

    del base_model_fp, adapter_model, merged_model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print("Freed fp16 model memory")
else:
    print("Merged model already exists at:", MERGED_MODEL_DIR)
    print("Nothing to clean up -- Cell 5 skipped loading fp16 model too.")

Merged model already exists at: /content/drive/MyDrive/legacyagent/checkpoints/qlora_v1_run2_eosfix/merged_model
Nothing to clean up -- Cell 5 skipped loading fp16 model too.


In [7]:
# CELL 7
# Reload the merged model with int8 quantization for the actual eval.
# Variable names match the frozen transcribe function (qwen_model, qwen_processor).
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_8bit=True)

qwen_processor = AutoProcessor.from_pretrained(str(MERGED_MODEL_DIR), trust_remote_code=True)

qwen_model = Qwen3ASRForConditionalGeneration.from_pretrained(
    str(MERGED_MODEL_DIR),
    quantization_config=quant_config,
    device_map="auto",
)
qwen_model.eval()

print("Quantized (int8) fine-tuned model loaded as qwen_model / qwen_processor")
print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Quantized (int8) fine-tuned model loaded as qwen_model / qwen_processor
GPU allocated: 2.2 GB


In [8]:
# CELL 8
# Output parser (verbatim from Day 2 Cell 10)
import re

def parse_qwen_asr_output(raw_output: str) -> str:
    text = raw_output.strip()
    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]
    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Parser loaded")

Parser loaded


In [9]:
# CELL 9
# Frozen inference function (verbatim from Day 6 Cell 6 -- no repetition_penalty,
# the EOS-fixed adapter does not need generation-time workarounds)
import torch

def transcribe_qwen_segment(row, waveform, sample_rate):
    start_sample = int(row["start"] * sample_rate)
    end_sample = int(row["end"] * sample_rate)
    audio_segment = waveform[:, start_sample:end_sample]
    audio_array = audio_segment.squeeze(0).numpy()

    conversation = [{"role": "user", "content": [{"type": "audio", "audio": audio_array}]}]

    inputs = qwen_processor.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )

    fixed_inputs = {}
    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):
            value = value.to(qwen_model.device)
            if value.is_floating_point():
                value = value.to(torch.bfloat16)
            fixed_inputs[key] = value
        else:
            fixed_inputs[key] = value

    with torch.inference_mode():
        generated_ids = qwen_model.generate(**fixed_inputs, max_new_tokens=256, do_sample=False)

    prompt_length = fixed_inputs["input_ids"].shape[1]
    generated_only = generated_ids[:, prompt_length:]
    raw_output = qwen_processor.batch_decode(generated_only, skip_special_tokens=True)[0].strip()
    transcript = parse_qwen_asr_output(raw_output)

    return raw_output, transcript

print("Frozen inference function loaded")

Frozen inference function loaded


In [10]:
# CELL 10
# Checkpointed full quantized inference over the frozen eval benchmark
import json, time
import torchaudio

QUANTIZED_RESULTS_PATH = RESULTS_DIR / "qwen3_asr_quantized_predictions.jsonl"

def load_existing_predictions(path):
    if not path.exists():
        return [], set()
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    completed = {r["segment_id"] for r in rows if r.get("status") == "success"}
    return rows, completed

def save_prediction(path, result):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

existing_rows, completed_ids = load_existing_predictions(QUANTIZED_RESULTS_PATH)
remaining_rows = [r for r in test_benchmark if r["segment_id"] not in completed_ids]

print("Total segments:", len(test_benchmark))
print("Already completed:", len(completed_ids))
print("Remaining:", len(remaining_rows))

cached_case_id = None
cached_waveform = None
cached_sample_rate = None
success_count = 0
failure_count = 0

for i, row in enumerate(remaining_rows, start=1):
    if row["case_id"] != cached_case_id:
        cached_waveform, cached_sample_rate = torchaudio.load(row["audio_path"])
        assert cached_sample_rate == 16000
        cached_case_id = row["case_id"]
        print(f"\nLoaded case: {cached_case_id}")

    t0 = time.time()
    try:
        raw_output, transcript = transcribe_qwen_segment(row, cached_waveform, cached_sample_rate)
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "case_name": row["case_name"], "start": row["start"], "end": row["end"],
            "duration": row["duration"], "reference": row["text"], "prediction": transcript,
            "raw_output": raw_output, "inference_seconds": round(time.time() - t0, 3),
            "model": "Qwen3-ASR-1.7B + QLoRA merged + int8", "status": "success",
        }
        success_count += 1
    except Exception as e:
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "reference": row["text"], "error": str(e), "status": "error",
        }
        failure_count += 1

    save_prediction(QUANTIZED_RESULTS_PATH, result)
    if i % 10 == 0:
        print(f"Processed {i}/{len(remaining_rows)}  (success={success_count}, failed={failure_count})")

print("\nDone. Success:", success_count, "Failed:", failure_count)

Total segments: 1163
Already completed: 1163
Remaining: 0

Done. Success: 0 Failed: 0


In [11]:
# CELL 11
# Official corpus WER for the quantized model
from jiwer import wer
import json

results = [json.loads(l) for l in open(QUANTIZED_RESULTS_PATH, encoding="utf-8") if l.strip()]
quantized_by_id = {r["segment_id"]: r for r in results if r.get("status") == "success"}

references, hypotheses = [], []
for row in test_benchmark:
    seg_id = row["segment_id"]
    if seg_id not in quantized_by_id:
        continue
    references.append(normalize_for_wer(row["text"]))
    hypotheses.append(normalize_for_wer(quantized_by_id[seg_id]["prediction"]))

quantized_wer = wer(references, hypotheses)
print("QUANTIZED (int8) FINE-TUNED QWEN3-ASR -- OFFICIAL RESULT")
print("=" * 60)
print("Evaluation segments:", len(references))
print("CORPUS WER (%)     :", round(quantized_wer * 100, 2))

QUANTIZED (int8) FINE-TUNED QWEN3-ASR -- OFFICIAL RESULT
Evaluation segments: 1163
CORPUS WER (%)     : 7.22


In [12]:
# CELL 12
# Entity-span WER for the quantized model (same frozen entity_terms.json)
import json

with open(EVAL_DIR / "entity_terms.json") as f:
    entity_terms_data = json.load(f)

candidate_terms = [t["term"].lower() for t in entity_terms_data["terms"]]
print("Entity vocabulary size:", len(candidate_terms))

def extract_entity_spans(text, terms):
    text_norm = normalize_for_wer(text)
    found = []
    for term in terms:
        term_norm = normalize_for_wer(term)
        if term_norm and term_norm in text_norm:
            found.append(term_norm)
    return found

entity_refs, entity_hyps = [], []
for row in test_benchmark:
    seg_id = row["segment_id"]
    if seg_id not in quantized_by_id:
        continue
    reference_text = row["text"]
    prediction_text = quantized_by_id[seg_id]["prediction"]

    ref_spans = extract_entity_spans(reference_text, candidate_terms)
    hyp_spans = extract_entity_spans(prediction_text, candidate_terms)
    if ref_spans:
        entity_refs.append(" ".join(ref_spans))
        entity_hyps.append(" ".join(hyp_spans) if hyp_spans else "")

entity_wer_quantized = wer(entity_refs, entity_hyps) if entity_refs else None
print("Quantized entity-span WER:", round(entity_wer_quantized * 100, 2) if entity_wer_quantized is not None else "n/a", "%")
print("Segments with entity mentions:", len(entity_refs))

Entity vocabulary size: 24
Quantized entity-span WER: 21.39 %
Segments with entity mentions: 163


In [13]:
# CELL 13
# Save Day 7 results and compare against the fine-tuned (non-quantized) numbers
import json
from datetime import datetime, timezone

# Fill these in from your actual Day 6 output before trusting this comparison
finetuned_overall_wer_pct = 7.36    # Qwen3-ASR + QLoRA, from Day 6
finetuned_entity_wer_pct = 22.99    # Qwen3-ASR + QLoRA, from Day 6

comparison = {
    "model": "Qwen3-ASR-1.7B + QLoRA (merged) + int8 post-training quantization",
    "overall_wer_pct": round(quantized_wer * 100, 2),
    "entity_span_wer_pct": round(entity_wer_quantized * 100, 2) if entity_wer_quantized is not None else None,
    "finetuned_overall_wer_pct": finetuned_overall_wer_pct,
    "finetuned_entity_wer_pct": finetuned_entity_wer_pct,
    "overall_wer_change_pct_points": round(quantized_wer * 100 - finetuned_overall_wer_pct, 2),
    "entity_wer_change_pct_points": (
        round(entity_wer_quantized * 100 - finetuned_entity_wer_pct, 2)
        if entity_wer_quantized is not None else None
    ),
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(comparison, indent=2))

with open(RESULTS_DIR / "day7_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\nSaved:", RESULTS_DIR / "day7_comparison.json")

{
  "model": "Qwen3-ASR-1.7B + QLoRA (merged) + int8 post-training quantization",
  "overall_wer_pct": 7.22,
  "entity_span_wer_pct": 21.39,
  "finetuned_overall_wer_pct": 7.36,
  "finetuned_entity_wer_pct": 22.99,
  "overall_wer_change_pct_points": -0.14,
  "entity_wer_change_pct_points": -1.6,
  "completed_at_utc": "2026-09-02T09:35:18.860037+00:00"
}

Saved: /content/drive/MyDrive/legacyagent/results/day7_quantization/day7_comparison.json
